In [1]:
# 2) 克隆仓库
import os
if not os.path.exists('/content/ops'):
    # TODO: replace with your fork URL after pushing
    !git clone https://github.com/hswsp/learn-cuda-from-scratch.git /content/ops
%cd /content/ops

Cloning into '/content/ops'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 239 (delta 62), reused 209 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (239/239), 267.74 KiB | 10.30 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/ops


In [3]:
# 3) 选择 GPU 架构
# Colab Free T4 → sm_75, A100 → sm_80, L4 → sm_89
import subprocess
out = subprocess.check_output('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True).decode().strip()
print('detected GPU:', out)
ARCH = 'sm_75' if 'T4' in out else ('sm_89' if 'L4' in out else 'sm_80')
print('using ARCH =', ARCH)

# 4) 编译并跑通第 5 章 memory hierarchy
!make -C code/ch05_memory ARCH={ARCH} clean all run

detected GPU: Tesla T4
using ARCH = sm_75
make: Entering directory '/content/ops/code/ch05_memory'
rm -f coalesce_vs_strided mem_modes shared_demo constant_demo
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr coalesce_vs_strided.cu -o coalesce_vs_strided
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr mem_modes.cu -o mem_modes
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr shared_demo.cu -o shared_demo
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr constant_demo.cu -o constant_demo
--- coalesce_vs_strided ---
N = 16777216 (64 MiB)
Pattern                      ms        BW
------------------------------------------
  coalesced (stride=1)         0.547 ms    245.5 GB/s
  strided 2                    0.767 ms    174.9 GB/s
  strided 4                    0.800 ms    167.7 GB/s
  strided 8                    0.814 ms    165.0 GB/s
  strided 32                   0.478 ms    280.8 GB/s

--- mem_modes ---
  device-on